# Ollama Model Inference

> Load and run ollama models


In [ ]:
# | default_exp llm


In [ ]:
# | exporti
import dotenv


In [ ]:
# | exporti
dotenv.load_dotenv()


In [ ]:
# | exporti
from typing import Literal, Optional
from ollama import list as ollama_list


In [ ]:
# | hide
from nbdev.showdoc import *


In [ ]:
# | export
def list_local_models():
  """List local models."""
  return [m.model for m in ollama_list()["models"]]


In [ ]:
list_local_models()


In [ ]:
# | exporti
from tqdm import tqdm
from ollama import pull


In [ ]:
# | exporti
def _download_model(model_name: str):
  """Download a model from ollama."""
  current_digest, bars = "", {}
  for progress in pull(model_name, stream=True):
    digest = progress.get("digest", "")
    if digest != current_digest and current_digest in bars:
      bars[current_digest].close()

    if not digest:
      print(progress.get("status"))
      continue

    if digest not in bars and (total := progress.get("total")):
      bars[digest] = tqdm(total=total, desc=f"pulling {digest[7:19]}", unit="B", unit_scale=True)

    if completed := progress.get("completed"):
      bars[digest].update(completed - bars[digest].n)

    current_digest = digest


In [ ]:
model_name = "gemma3:27b"
local_models = list_local_models()
if model_name not in local_models:
  _download_model(model_name)


In [ ]:
# | exporti
from ollama import chat as ollama_chat


In [ ]:
ollama_response = ollama_chat(
  model=model_name, think=None, messages=[{"role": "user", "content": "Merhaba. Nasılsın?"}]
)
ollama_response.message.content, ollama_response.message.thinking


In [ ]:
# | export
def create_message(content: Optional[str], role: Literal["user", "assistant", "system"] = "user"):
  """Create a message for the chat."""
  return {"role": role, "content": content}


In [ ]:
# | exporti
def _chat_one(model: str, message: str, think: Optional[bool] = None, **kwargs):
  """Chat with the model."""
  return ollama_chat(
    model=model, think=think, messages=[create_message(message)], **kwargs
  ).message.content


In [ ]:
_chat_one("gemma3:27b", "Merhaba. Nasılsın?")


In [ ]:
# | exporti
from pydantic import BaseModel


In [ ]:
# | export
class OutputSchema(BaseModel):
  """Base class for structured LLM outputs. Subclass this to define your schema.

  The `content` field is required and will be used for chat history."""

  content: str


In [ ]:
class _Response(OutputSchema):
  """Example response schema."""

  reasoning: str
  is_happy: bool


In [ ]:
_Response.model_validate_json(
  _chat_one(model_name, "Mulu musun?", think=None, format=_Response.model_json_schema())
)


# LLM Cloud


In [ ]:
# | exporti
from ollama import ChatResponse
from litellm import completion as litellm_completion


In [ ]:
model = "openrouter/google/gemini-2.5-flash-lite-preview-09-2025"


In [ ]:
response = litellm_completion(
  model=model,
  messages=[{"role": "user", "content": "Merhaba. Nasılsın?"}],
)
response.choices[0].message.content


In [ ]:
response = litellm_completion(
  model=model,
  messages=[create_message("Hello, how are you?")],
  response_format=_Response,
)


In [ ]:
_Response.model_validate_json(response.choices[0].message.content)


In [ ]:
# | export
class LLM:
  """A class for interacting with an LLM."""

  def __init__(
    self,
    model_name: str,  # The name of the model to use
    system_prompt: str = "",  # The system prompt to use.
    think: Optional[bool] = None,  # Whether to enable thinking mode
    output_schema: Optional[
      type[OutputSchema]
    ] = None,  # Schema for structured output (must subclass OutputSchema)
  ):
    self.model_name = model_name
    self.system_prompt = system_prompt
    self.output_schema = output_schema
    self.messages: list[dict] = [create_message(system_prompt, "system")]
    self.think = think
    self.use_openrouter = "openrouter" in model_name

    if not self.use_openrouter:
      self._verify()

  def generate(self, message: str, **kwargs) -> str | OutputSchema:
    """One-shot generation. Does not maintain conversation history."""
    messages = [create_message(self.system_prompt, "system"), create_message(message, "user")]
    if self.use_openrouter:
      return self._cloud_call(messages, **kwargs)
    return self._local_call(messages, **kwargs)

  def chat(self, message: str, **kwargs) -> str | OutputSchema:
    """Multi-turn chat. Maintains conversation history."""
    self.messages.append(create_message(message, "user"))

    if self.use_openrouter:
      response = self._cloud_call(self.messages, **kwargs)
    else:
      response = self._local_call(self.messages, **kwargs)

    # Extract content for history: schema responses have .content, plain responses are strings
    if isinstance(response, OutputSchema):
      self.messages.append(create_message(response.content, "assistant"))
    else:
      self.messages.append(create_message(response, "assistant"))
    return response

  def _local_call(self, messages: list[dict], **kwargs) -> str | OutputSchema:
    """Call local Ollama model."""
    response: ChatResponse = ollama_chat(
      model=self.model_name,
      think=self.think,
      messages=messages,
      format=self.output_schema.model_json_schema() if self.output_schema else None,
      **kwargs,
    )
    return self._parse_response(response.message.content)

  def _cloud_call(self, messages: list[dict], **kwargs) -> str | OutputSchema:
    """Call cloud model via litellm."""
    if self.think is True:
      kwargs["reasoning_effort"] = "low"
    response = litellm_completion(
      model=self.model_name,
      messages=messages,
      response_format=self.output_schema if self.output_schema else None,
      **kwargs,
    )
    return self._parse_response(response.choices[0].message.content)

  def _parse_response(self, content: Optional[str]) -> str | OutputSchema:
    """Parse raw response content, validating against schema if set."""
    assert content is not None, "No content was returned"
    if self.output_schema:
      return self.output_schema.model_validate_json(content)
    return content

  async def agenerate_batch(
    self,
    messages: list[str],
    concurrency: int = 10,
    show_progress: bool = True,
    **kwargs,
  ) -> list[str | OutputSchema]:
    """Batch async generation with concurrency control.

    Args:
        messages: List of input messages to process
        concurrency: Max parallel requests (default 10)
        show_progress: Show tqdm progress bar
        **kwargs: Additional args passed to the API (e.g., temperature)

    Returns:
        List of responses in same order as inputs
    """
    import asyncio
    from tqdm.asyncio import tqdm_asyncio

    semaphore = asyncio.Semaphore(concurrency)

    if self.use_openrouter:
      from litellm import acompletion

      async def process_one(message: str) -> str | OutputSchema:
        async with semaphore:
          msgs = [create_message(self.system_prompt, "system"), create_message(message, "user")]
          response = await acompletion(
            model=self.model_name,
            messages=msgs,
            response_format=self.output_schema if self.output_schema else None,
            **kwargs,
          )
          return self._parse_response(response.choices[0].message.content)
    else:
      from ollama import AsyncClient

      async def process_one(message: str) -> str | OutputSchema:
        async with semaphore:
          msgs = [create_message(self.system_prompt, "system"), create_message(message, "user")]
          client = AsyncClient()
          response = await client.chat(
            model=self.model_name,
            messages=msgs,
            format=self.output_schema.model_json_schema() if self.output_schema else None,
            **kwargs,
          )
          return self._parse_response(response.message.content)

    tasks = [process_one(msg) for msg in messages]
    if show_progress:
      return await tqdm_asyncio.gather(*tasks, desc="Processing")
    return await asyncio.gather(*tasks)

  def _verify(self):
    """Verify the model is available, download if missing."""
    if self.model_name not in list_local_models():
      _download_model(self.model_name)


# Tests


In [ ]:
# Test create_message with different roles
assert create_message("hello") == {"role": "user", "content": "hello"}
assert create_message("hello", "assistant") == {"role": "assistant", "content": "hello"}
assert create_message("system prompt", "system") == {"role": "system", "content": "system prompt"}
assert create_message(None) == {"role": "user", "content": None}
print("✓ create_message tests passed")


In [ ]:
# Test OutputSchema validation
class TestSchema(OutputSchema):
  sentiment: str

# Valid JSON should parse correctly
parsed = TestSchema.model_validate_json('{"content": "hello", "sentiment": "positive"}')
assert parsed.content == "hello"
assert parsed.sentiment == "positive"

# content field is required
from pydantic import ValidationError
try:
  TestSchema.model_validate_json('{"sentiment": "positive"}')
  assert False, "Should have raised ValidationError"
except ValidationError:
  pass

print("✓ OutputSchema tests passed")


In [ ]:
# Test LLM initialization and routing logic
# OpenRouter models should set use_openrouter=True
llm_cloud = LLM("openrouter/google/gemini-2.5-flash", system_prompt="Be helpful")
assert llm_cloud.use_openrouter == True
assert llm_cloud.model_name == "openrouter/google/gemini-2.5-flash"
assert llm_cloud.system_prompt == "Be helpful"
assert llm_cloud.messages == [{"role": "system", "content": "Be helpful"}]

print("✓ LLM initialization tests passed")


In [ ]:
# Test list_local_models returns a list of strings
models = list_local_models()
assert isinstance(models, list)
assert all(isinstance(m, str) for m in models)
print(f"✓ list_local_models tests passed (found {len(models)} models)")


In [ ]:
# Test LLM._parse_response without schema (returns string as-is)
llm_no_schema = LLM("openrouter/test", output_schema=None)
assert llm_no_schema._parse_response("hello world") == "hello world"

# Test LLM._parse_response with schema
llm_with_schema = LLM("openrouter/test", output_schema=TestSchema)
result = llm_with_schema._parse_response('{"content": "test", "sentiment": "neutral"}')
assert isinstance(result, TestSchema)
assert result.content == "test"
assert result.sentiment == "neutral"

print("✓ LLM._parse_response tests passed")


In [ ]:
# Test agenerate_batch works with local Ollama models
import asyncio
async def test_agenerate_batch_local():
  llm_local = LLM("gemma3:27b")
  results = await llm_local.agenerate_batch(["Say 'test' and nothing else"], show_progress=False)
  assert isinstance(results, list)
  assert len(results) == 1
  return True

assert asyncio.run(test_agenerate_batch_local())
print("✓ agenerate_batch local model test passed")


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()
